In [1]:
# Import Required Python Libraries
import os, json, time, random
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

Configuration 

In [2]:
gpt_data = "C:\\Users\\Duneland\\Documents\\ITS 520 Project\\gpt_train.csv"  
# Modify to actual location when necessary

output_dir = "artifacts"

In [3]:
# Target to predict
target_column = "val_loss"

log_target = False

In [4]:
# Hyperparameters 
n_model = 5
epochs = 40
batch_size = 512
max_lr = 2e-3
weight_decay = 1e-4
hidden = 256
depth = 5 # number of hidden layers
seed = 42

In [5]:
# Make Output directory and check for GPU
os.makedirs(output_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

Device: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [6]:
# Function 
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(seed)

Load Data

In [7]:

df = pd.read_csv(gpt_data)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded 100,000 rows x 30 columns


Data Preprocessing

In [8]:
# Data Preprocessing
id_cols = ["run_id", "seed"]
for c in df.columns:                                                     # Loop through every column name in the DataFrame
    if df[c].dtype == bool:                                              # Check if the column has a boolean data type
        df[c] = df[c].astype(int)                                        # Convert True/False values to 1/0 integers
        print(f"  [bool -> int]   {c}")                                  # Log the converted column name

string_cols = [c for c in df.columns if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]  # Collect all object or string-type columns
for c in string_cols:                                                    # Loop through each string/object column
    n = df[c].nunique()                                                  # Count the number of unique values in the column
    if c in id_cols:                                                     # Check if the column is a predefined ID column
        print(f"  [drop ID]       {c} ({n} unique)")                     # Log that the ID column will be dropped
    elif n <= 1:                                                         # Check if the column has 0 or 1 unique value (constant/empty)
        print(f"  [drop constant] {c} = {df[c].iloc[0]!r}")              # Log the constant column and its first value
    else:                                                                # Otherwise, treat it as a categorical feature
        print(f"  [one-hot]       {c} ({n} categories)")                 # Log that the column will be one-hot encoded
        df = pd.concat([df, pd.get_dummies(df[c], prefix=c, dtype=int)], axis=1)  # Append one-hot encoded 0/1 columns to the DataFrame
    df = df.drop(columns=c)                                              # Drop the original string column 


  [bool -> int]   bias
  [drop ID]       run_id (5000 unique)
  [drop constant] device = 'NVIDIA GeForce RTX 5070 Ti'
  [drop constant] dataset_id = 'Salesforce/wikitext:wikitext-2-raw-v1'
  [drop constant] torch_version = '2.11.0+cu128'


Feature Selection 

In [9]:
# measured after training: would leak the target
# val_perplexity = exp(val_loss) exactly
outcome_cols = [  
    "train_loss", "val_loss", "val_perplexity",   
    "train_seconds", "eval_seconds", "cumulative_seconds", "peak_gpu_memory_mb"]

In [10]:
# Mathematical functions of other inputs that are redundant for this regression model
derived_cols = [                   # Define a list of redundant columns computed from other inputs
    "mlp_hidden_dim",              # Derived from embed_dim * mlp_ratio
    "parameter_count",             # Derived from layers, embed_dim, mlp_ratio, and bias
    "optimizer_steps_epoch",       # Derived from ceil(1953 / batch_size)
    "cumulative_optimizer_steps",  # Derived from epoch * optimizer_steps_epoch
    "cumulative_train_tokens"]     # Derived from epoch * 499,968

constant_numeric = [c for c in df.columns if df[c].nunique() <= 1]  # Collect remaining columns with 0 or 1 unique value
print("  [drop constant numeric]", constant_numeric)                # Log the constant numeric columns to be dropped

  [drop constant numeric] ['seq_len', 'train_windows', 'val_windows', 'epoch_train_tokens']


In [11]:
# Learning rate spans 1e-4..5e-4 multiplicatively: model it in base-10 logarithmic scale
df["log10_learning_rate"] = np.log10(df["learning_rate"])

In [12]:
# Input features (Dataframe) to convert into 32-bit NumPy arrays for training
features = ["layers", "embed_dim", "heads", "mlp_ratio", "bias",
            "log10_learning_rate", "batch_size", "dropout", "epoch"]
# Append any one-hot encoded columns from string columns
features += [c for c in df.columns if any(c.startswith(s + "_") for s in string_cols)]  
config_cols = ["layers", "embed_dim", "heads", "mlp_ratio", "bias",
               "learning_rate", "batch_size", "dropout"]

print("Features:", features)

X = df[features].values.astype(np.float32)     # Extract as 32-bit float NumPy array
y_raw = df[target_column].values.astype(np.float32) # Extract as 32-bit float NumPy array
y_t = np.log(y_raw) if log_target else y_raw   # Apply natural log to target 

Features: ['layers', 'embed_dim', 'heads', 'mlp_ratio', 'bias', 'log10_learning_rate', 'batch_size', 'dropout', 'epoch']


Split by Configuration 

In [13]:
# Group Train/Val/Test Split (70/15/15)
groups = df[config_cols].astype(str).agg("|".join, axis=1).values                         # Join config columns into a single string ID per row
idx = np.arange(len(df))                                                                  # Create an array of row indices from 0 to len(df) - 1
trval_idx, test_idx = next(GroupShuffleSplit(1, test_size=0.15, random_state=seed)        # Split 15% of unique configs into test, 85% into train+val
                           .split(idx, groups=groups))                                    # Pass row indices and config group IDs to the splitter
tr_sub, va_sub = next(GroupShuffleSplit(1, test_size=0.15 / 0.85, random_state=seed)      # Split the remaining 85% so val is 15% of the total dataset
                      .split(trval_idx, groups=groups[trval_idx]))                        # Split using only the train+val subset and its group IDs
train_idx, val_idx = trval_idx[tr_sub], trval_idx[va_sub]                                 # Map relative subset positions back to original DataFrame indices
print(f"Split rows  train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}") # Print total row counts for train, val, and test splits
print(f"Split configs train={len(set(groups[train_idx]))} val={len(set(groups[val_idx]))} " # Print unique config counts for train and val
      f"test={len(set(groups[test_idx]))}")                                               # Print unique config count for test

# Measure Baseline noise 
_pair = df.assign(g=groups, yt=y_t).groupby(["g", "epoch"])["yt"].agg(["first", "last", "size"])  # Group by config and epoch to get first/last seed targets and count
_pair = _pair[_pair["size"] == 2]                                                         # Keep only (config, epoch) pairs that have both seeds
noise_rmse = float(np.sqrt(((_pair["first"] - _pair["last"]) ** 2).mean() / 2))           # Estimate irreducible seed-to-seed noise standard deviation (RMSE)
noise_r2 = float(1 - noise_rmse ** 2 / y_t.var())                                         # Convert the noise floor into the maximum theoretical R2 score
print(f"Seed-noise floor: RMSE={noise_rmse:.5f}  ->  best achievable R2 ~ {noise_r2:.5f}") # Print the irreducible noise RMSE and ceiling R2

Split rows  train=70,000 val=15,000 test=15,000
Split configs train=1750 val=375 test=375
Seed-noise floor: RMSE=0.01593  ->  best achievable R2 ~ 0.99707


Scaling (Train Only)

In [14]:
x_scaler = StandardScaler().fit(X[train_idx])                                         # Fit feature scaler on training data only (prevents leakage)
y_mean, y_std = float(y_t[train_idx].mean()), float(y_t[train_idx].std())             # Compute mean and std of training targets for standardization
joblib.dump(x_scaler, os.path.join(output_dir, "scaler.pkl"))                         # Save the fitted scaler to disk for later inference
print("Saved scaler.pkl")                                                             # Log that the scaler file was saved

Xs = torch.tensor(x_scaler.transform(X), dtype=torch.float32, device=device)          # Scale all features and convert to a float32 tensor on device
Ys = torch.tensor((y_t - y_mean) / y_std, dtype=torch.float32, device=device)[:, None] # Scale all targets, convert to float32 tensor, and add column dim (N, 1)
tr_t = torch.tensor(train_idx, device=device)                                         # Move training row indices to a tensor on device
va_t = torch.tensor(val_idx, device=device)                                           # Move validation row indices to a tensor on device

Saved scaler.pkl


Transformer Model 

In [15]:
#  MLP on standardized inputs. SiLU (smooth) instead of ReLU: it's more accurate here
#    and gives NIO smooth, non-zero input gradients everywhere.
class TransformerLossRegressor(nn.Module):                                             # Define an MLP regression model inheriting from nn.Module
   
    def __init__(self, input_dim, hidden=hidden, depth=depth):                         # Initialize with input size, hidden width, and number of layers
        super().__init__()                                                             # Initialize the parent nn.Module class
        layers, d = [], input_dim                                                      # Start an empty layer list and track current layer input size
        for _ in range(depth):                                                         # Repeat for each hidden layer
            layers += [nn.Linear(d, hidden), nn.SiLU()]                                # Add a linear layer followed by a smooth SiLU activation
            d = hidden                                                                 # Update input dimension for the next layer
        layers.append(nn.Linear(d, 1))                                                 # Add final linear layer outputting a single scalar prediction
        self.net = nn.Sequential(*layers)                                              # Pack all layers into a sequential container

    def forward(self, x):                                                              # Define the forward pass
        return self.net(x)                                                             # Run input x through the sequential network


                   

In [16]:
# It bundles the ensemble models and the input/output scaling into a single PyTorch module, 
# allowing gradients to flow straight from the output back to the raw inputs.
class NIOSurrogate(nn.Module):                                                         # Define a differentiable ensemble wrapper with built-in scaling
    
    def __init__(self, input_dim, n_models, x_mean, x_std, y_mean, y_std, log_target=False):  # Initialize ensemble size, scaling stats, and log flag
        super().__init__()                                                             # Initialize the parent nn.Module class
        self.members = nn.ModuleList([TransformerLossRegressor(input_dim) for _ in range(n_models)])  # Create a list of n_models independent regressors
        self.register_buffer("x_mean", torch.as_tensor(x_mean, dtype=torch.float32))   # Store feature means as a non-trainable tensor buffer
        self.register_buffer("x_std", torch.as_tensor(x_std, dtype=torch.float32))     # Store feature stds as a non-trainable tensor buffer
        self.register_buffer("y_mean", torch.tensor(y_mean, dtype=torch.float32))      # Store target mean as a non-trainable tensor buffer
        self.register_buffer("y_std", torch.tensor(y_std, dtype=torch.float32))        # Store target std as a non-trainable tensor buffer
        self.log_target = log_target                                                   # Save whether predictions need exp() to undo log transform

    def forward_scaled(self, xs):                                                      # Predict directly on already-standardized inputs
        return torch.stack([m(xs) for m in self.members]).mean(0)                      # Stack all member outputs and average across the ensemble

    def forward(self, x_raw, return_std=False):                                        # Predict on raw inputs, optionally returning ensemble uncertainty
        xs = (x_raw - self.x_mean) / self.x_std                                        # Standardize raw inputs using the stored training mean and std
        preds = torch.stack([m(xs) for m in self.members]) * self.y_std + self.y_mean  # Run all models and unscale their predictions back to y_t units
        if self.log_target:                                                            # Check if the target was log-transformed during training
            preds = torch.exp(preds)                                                   # Exponentiate predictions to return to original target units
        mean = preds.mean(0)                                                           # Compute the mean prediction across all ensemble members
        return (mean, preds.std(0)) if return_std else mean       

Training 

In [17]:
def train_member(seed):                                                                # Define function to train one ensemble member with a given seed
    set_seed(seed)                                                                     # Set random seeds for reproducible weight init and shuffling
    model = TransformerLossRegressor(len(features)).to(device)                         # Initialize the MLP regressor and move it to the target device
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)  # Set up AdamW optimizer with weight decay
    # Calculate number of mini-batch steps per epoch
    steps_per_epoch = int(np.ceil(len(train_idx) / batch_size))                        
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=epochs * steps_per_epoch)  # Configure OneCycleLR scheduler across all steps
    loss_fn = nn.MSELoss()                                                             # Use Mean Squared Error as the regression loss function
    best_val, best_state, best_ep = float("inf"), None, 0                              # Track best validation loss, weights, and epoch

    for ep in range(epochs):                                                           # Loop through each training epoch
        model.train()                                                                  # Put model in training mode
        perm = tr_t[torch.randperm(len(tr_t), device=device)]                          # Shuffle training indices on-device for this epoch
        tl = 0.0                                                                       # Reset running training loss sum
        for k in range(0, len(perm), batch_size):                                      # Iterate through shuffled indices in mini-batches
            b = perm[k:k + batch_size]                                                 # Slice current batch of row indices
            opt.zero_grad(set_to_none=True)                                            # Clear previous gradients efficiently
            loss = loss_fn(model(Xs[b]), Ys[b])                                        # Compute MSE loss on the standardized batch
            loss.backward()                                                            # Backpropagate gradients through the model
            opt.step(); sched.step()                                                   # Update weights and advance the learning rate schedule
            tl += loss.item() * len(b)                                                 # Accumulate total loss weighted by batch size

        model.eval()                                                                   # Put model in evaluation mode
        with torch.no_grad():                                                          # Disable gradient tracking for validation
            pv = model(Xs[va_t])                                                       # Run forward pass on all validation features
            vl = loss_fn(pv, Ys[va_t]).item()                                          # Compute scalar validation MSE
            vr2 = 1 - vl / Ys[va_t].var(unbiased=False).item()                         # Compute validation R2 score
        if vl < best_val:                                                              # Check if validation loss improved
            best_val, best_ep = vl, ep + 1                                             # Update best validation loss and epoch number
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # Clone and save the current best model weights
        if (ep + 1) % 5 == 0 or ep == 0:                                               # Log metrics on epoch 1 and every 5th epoch
            print(f"  [model {seed}] Epoch {ep+1:3d}/{epochs} | Train MSE(scaled) {tl/len(perm):.5f} | "  # Print epoch and average training MSE
                  f"Val MSE(scaled) {vl:.5f} | Val R2 {vr2:.5f}")                      # Print validation MSE and R2

    model.load_state_dict(best_state)                                                  # Restore the best validation weights before returning
    print(f"  [model {seed}] best epoch {best_ep}, val MSE(scaled) {best_val:.5f}")    # Log the best epoch and its validation MSE
    return model                                                                       # Return the trained model


surrogate = NIOSurrogate(len(features), n_model, x_scaler.mean_, x_scaler.scale_,      # Instantiate the ensemble surrogate with fitted feature scaling
                         y_mean, y_std, log_target).to(device)                         # Pass target scaling stats, log flag, and move to device
t0 = time.time()                                                                       # Record start time for total training duration
for i in range(n_model):                                                               # Loop over each ensemble member slot
    print(f"\nTraining ensemble member {i+1}/{n_model}")                               # Log which ensemble member is being trained
    surrogate.members[i].load_state_dict(train_member(seed + i).state_dict())          # Train member with a unique seed and copy its weights into surrogate
print(f"\nTraining time: {time.time()-t0:.1f}s")                                       # Print total elapsed training time in seconds


Training ensemble member 1/5
  [model 42] Epoch   1/40 | Train MSE(scaled) 0.51750 | Val MSE(scaled) 0.15750 | Val R2 0.84154
  [model 42] Epoch   5/40 | Train MSE(scaled) 0.02264 | Val MSE(scaled) 0.02012 | Val R2 0.97976
  [model 42] Epoch  10/40 | Train MSE(scaled) 0.00880 | Val MSE(scaled) 0.01080 | Val R2 0.98914
  [model 42] Epoch  15/40 | Train MSE(scaled) 0.00587 | Val MSE(scaled) 0.00770 | Val R2 0.99226
  [model 42] Epoch  20/40 | Train MSE(scaled) 0.00441 | Val MSE(scaled) 0.00509 | Val R2 0.99488
  [model 42] Epoch  25/40 | Train MSE(scaled) 0.00388 | Val MSE(scaled) 0.00469 | Val R2 0.99528
  [model 42] Epoch  30/40 | Train MSE(scaled) 0.00339 | Val MSE(scaled) 0.00445 | Val R2 0.99553
  [model 42] Epoch  35/40 | Train MSE(scaled) 0.00306 | Val MSE(scaled) 0.00437 | Val R2 0.99560
  [model 42] Epoch  40/40 | Train MSE(scaled) 0.00294 | Val MSE(scaled) 0.00425 | Val R2 0.99572
  [model 42] best epoch 37, val MSE(scaled) 0.00423

Training ensemble member 2/5
  [model 43] Ep

Evaluation

In [18]:
surrogate.eval()                                                                       # Put the surrogate ensemble in evaluation mode
with torch.no_grad():                                                                  # Disable gradient tracking for inference
    X_raw_t = torch.tensor(X, device=device)                                           # Convert raw feature matrix X to a tensor on device
    pred_all = surrogate(X_raw_t).cpu().numpy().ravel()                                # Get ensemble predictions in original units as a 1D NumPy array
    member_preds = [(m(Xs).cpu().numpy().ravel() * y_std + y_mean) for m in surrogate.members]  # Get each individual member's unscaled predictions on Xs
if log_target:                                                                         # Check if the target was log-transformed during training
    member_preds = [np.exp(p) for p in member_preds]                                   # Exponentiate each member's predictions back to original units


def metrics(idx_, pred):                                                               # Define helper to compute regression metrics on a given index subset
    yt, p = y_raw[idx_], pred[idx_]                                                    # Slice ground-truth targets and predictions for those indices
    return {"r2": float(r2_score(yt, p)), "rmse": float(mean_squared_error(yt, p) ** 0.5),  # Return R2 and RMSE as Python floats
            "mae": float(mean_absolute_error(yt, p))}                                  # Include MAE as a Python float


m_train, m_val, m_test = metrics(train_idx, pred_all), metrics(val_idx, pred_all), metrics(test_idx, pred_all)  # Evaluate ensemble on train, val, and test splits
single_test = [metrics(test_idx, p) for p in member_preds]                             # Evaluate each individual ensemble member on the test split
final_epoch_test = test_idx[df["epoch"].values[test_idx] == df["epoch"].max()]         # Filter test indices to only rows at the final (max) epoch
m_final_epoch = metrics(final_epoch_test, pred_all)                                    # Evaluate ensemble performance specifically on the final epoch
per_epoch_rmse = {int(e): float(np.sqrt(np.mean((y_raw[test_idx][df["epoch"].values[test_idx] == e]  # Compute test RMSE for each unique epoch
- pred_all[test_idx][df["epoch"].values[test_idx] == e]) ** 2)))                       # Subtract predictions from ground truth, square, mean, and sqrt
                  for e in sorted(df["epoch"].unique())}                               # Iterate over all epochs in ascending order

Results

In [19]:
print("\n================ RESULTS (target: %s) ================" % target_column)  # Print results header
print(f"Train  R2 {m_train['r2']:.5f} | RMSE {m_train['rmse']:.5f}")                # Log train R2 and RMSE
print(f"Val    R2 {m_val['r2']:.5f} | RMSE {m_val['rmse']:.5f}")                    # Log validation R2 and RMSE
print(f"TEST   R2 {m_test['r2']:.5f} | RMSE {m_test['rmse']:.5f} | MAE {m_test['mae']:.5f}   <- unseen configs")  # Log test metrics on unseen configs
print(f"Single-model test RMSE: {[round(s['rmse'], 5) for s in single_test]}")      # Log each individual ensemble member's test RMSE
print(f"Test at final epoch only: R2 {m_final_epoch['r2']:.5f} | RMSE {m_final_epoch['rmse']:.5f}")  # Log test metrics at the final epoch
if not log_target:
    print(f"Noise floor (seed variance): RMSE {noise_rmse:.5f} | max R2 ~ {noise_r2:.5f}")  # Log irreducible noise floor (when in raw target units)

try:
    import matplotlib
    matplotlib.use("Agg")                                                           # Use non-interactive backend for saving plots without a display
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))                                   # Create a 1x2 figure for side-by-side evaluation plots

    ax[0].scatter(y_raw[test_idx], pred_all[test_idx], s=2, alpha=0.3)              # Plot actual vs. predicted test values
    lo, hi = y_raw.min(), y_raw.max()
    ax[0].plot([lo, hi], [lo, hi], "k--", lw=1)                                     # Draw y = x reference line for perfect predictions
    ax[0].set_xlabel(f"Actual {target_column}"); ax[0].set_ylabel(f"Predicted {target_column}")
    ax[0].set_title(f"Test set (unseen configs)  R² = {m_test['r2']:.4f}")

    ax[1].plot(list(per_epoch_rmse), list(per_epoch_rmse.values()), "o-", label="Model test RMSE")  # Plot test RMSE across training epochs
    if not log_target:
        noise_by_ep = _pair.reset_index().assign(d2=lambda t: (t["first"] - t["last"]) ** 2 / 2) \
                           .groupby("epoch")["d2"].mean() ** 0.5                    # Compute per-epoch seed-to-seed noise RMSE
        ax[1].plot(noise_by_ep.index, noise_by_ep.values, "s--", label="Seed-noise floor")  # Overlay the noise floor curve for comparison
    ax[1].set_xlabel("Training epoch"); ax[1].set_ylabel("RMSE"); ax[1].legend()
    ax[1].set_title("Error by epoch")

    plt.tight_layout(); plt.savefig(os.path.join(output_dir, "evaluation.png"), dpi=130)  # Save the combined figure to disk
    print("Saved evaluation.png")
except ImportError:
    pass                                                                            # Skip plotting if matplotlib is not installed


================ RESULTS (target: val_loss) ================
Train  R2 0.99710 | RMSE 0.01585
Val    R2 0.99590 | RMSE 0.01880
TEST   R2 0.99621 | RMSE 0.01807 | MAE 0.01170   <- unseen configs
Single-model test RMSE: [0.01837, 0.01858, 0.01815, 0.01849, 0.01815]
Test at final epoch only: R2 0.98560 | RMSE 0.02530
Noise floor (seed variance): RMSE 0.01593 | max R2 ~ 0.99707
Saved evaluation.png


Save Artifacts

In [20]:
torch.save(surrogate.state_dict(), os.path.join(output_dir, "model_v1.pt"))  # Save trained ensemble weights and scaling buffers to disk
print("Saved model_v1.pt")

input_space = {}                                                             # Store valid min/max bounds and discrete values for NIO
for f in features:
    vals = np.sort(df[f].unique())                                           # Get sorted unique values for each feature
    input_space[f] = {"min": float(vals.min()), "max": float(vals.max()),    # Record feature bounds
                      "allowed_values": [float(v) for v in vals] if len(vals) <= 25 else None}  # Save discrete choices if <= 25 unique values

metadata = {                                                                 # Bundle all info needed to reload the model and run NIO
    "feature_order": features,                                               # Exact input column order expected by the model
    "target_name": target_column,
    "log_target": log_target,
    "input_dim": len(features),
    "scaled": True,
    "scaler_type": "StandardScaler (also embedded as buffers inside model_v1.pt)",
    "x_mean": x_scaler.mean_.tolist(),                                       # Save feature scaling means as a JSON-serializable list
    "x_std": x_scaler.scale_.tolist(),                                       # Save feature scaling stds as a JSON-serializable list
    "y_mean": y_mean, "y_std": y_std,
    "model": {"class": "NIOSurrogate", "n_models": n_model, "hidden": hidden, "depth": depth,
              "activation": "SiLU"},                                         # Record architecture hyperparameters for reconstruction
    "input_transforms": {"log10_learning_rate": "log10(learning_rate)", "bias": "True->1, False->0"},
    "input_space": input_space,                                              # Attach feature bounds and allowed discrete values
    "constraints": [                                                         # Document domain constraints NIO must respect
        "embed_dim % heads == 0 (only these head counts appear in the data)",
        "layers, embed_dim, heads, mlp_ratio, batch_size, epoch are integers; bias is 0/1",
        "stay inside min/max of the training data: the surrogate is not reliable outside it"],
    "parameter_count_formula": "layers*(4*d^2 + 2*d*h + 2*d) + 513*d + bias*(layers*(7*d + h) + d), "
                               "d=embed_dim, h=embed_dim*mlp_ratio",         # Closed-form parameter count for multi-objective NIO
    "split": "GroupShuffleSplit by hyperparameter configuration, 70/15/15",
}
with open(os.path.join(output_dir, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)                                         # Write metadata dictionary to metadata.json
print("Saved metadata.json")

final_metrics = {                                                            # Collect all evaluation metrics and dataset split counts
    "target": target_column,
    "test": m_test, "validation": m_val, "train": m_train,
    "test_final_epoch_only": m_final_epoch,
    "test_single_models": single_test,
    "test_rmse_by_epoch": per_epoch_rmse,
    "noise_floor_rmse": noise_rmse if not log_target else None,              # Include noise floor metrics only when in raw target units
    "noise_floor_max_r2": noise_r2 if not log_target else None,
    "n_models": n_model, "epochs_per_model": epochs,
    "n_train_rows": int(len(train_idx)), "n_val_rows": int(len(val_idx)), "n_test_rows": int(len(test_idx)),
}
with open(os.path.join(output_dir, "metrics.json"), "w") as f:
    json.dump(final_metrics, f, indent=4)                                    # Write evaluation metrics to metrics.json
print("Saved metrics.json")

Saved model_v1.pt
Saved metadata.json
Saved metrics.json
